 COLAB SETUP

In [ ]:
# -----------------------------------------------------------
# CELL 1: SETUP & MOUNT DRIVE
# -----------------------------------------------------------
from google.colab import drive
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# 1. Mount Drive
if os.path.ismount('/content/drive'):
    print("✓ Drive already mounted.")
else:
    drive.mount('/content/drive', force_remount=True)

# 2. Define Paths
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
DATA_DIR = f"{BASE_DIR}/dataset"
MODEL_DIR = f"{BASE_DIR}/models"
FEATURE_DIR = f"{BASE_DIR}/features"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

print("✓ Directories ready.")

Mounted at /content/drive
✓ Directories ready.


 Manual Image Processing

In [ ]:
# -----------------------------------------------------------
# CELL 2: MANUAL IMAGE PROCESSING (NO TORCHVISION)
# -----------------------------------------------------------
def manual_process_image(image_path):
    """
    Reads, Resizes, and Normalizes an image using NumPy math.
    """
    try:
        # 1. Open Image
        img = Image.open(image_path).convert('RGB')

        # 2. Manual Resize to 224x224 (PIL is allowed for basic I/O)
        img = img.resize((224, 224))

        # 3. Convert to NumPy Array & Scale to [0, 1]
        img_np = np.array(img, dtype=np.float32) / 255.0

        # 4. Manual Normalization (Standard ImageNet stats)
        # Formula: (pixel - mean) / std
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

        # Math: (Image - Mean) / Std
        img_np = (img_np - mean) / std

        # 5. Transpose: (Height, Width, Channel) -> (Channel, Height, Width)
        # Required because the Pre-trained model expects channels first
        img_np = img_np.transpose(2, 0, 1)

        return img_np
    except Exception as e:
        return None

print("✓ Manual processing function defined.")

✓ Manual processing function defined.


Feature Extraction (Using Pre-trained Weights)

In [ ]:
# -----------------------------------------------------------
# CELL 3: EXTRACT FEATURES (USING PRE-TRAINED MODEL)
# -----------------------------------------------------------
import torch
import torchvision.models as models

# 1. Load Pre-trained Model
print("Loading ResNet50 Weights...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(weights="IMAGENET1K_V1")
model.fc = torch.nn.Identity() # Remove the last layer
model = model.to(device)
model.eval()

def extract_features_manual(root_dir):
    if not os.path.exists(root_dir):
        print("Data directory not found.")
        return np.array([]), np.array([])

    classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    ignore = ['Fabric Defects Dataset', 'fabricfaultdetection']
    classes = [c for c in classes if c not in ignore]

    print(f"Classes found: {classes}")

    features = []
    labels = [] # Strings

    for cls_name in classes:
        cls_path = os.path.join(root_dir, cls_name)
        file_names = os.listdir(cls_path)
        print(f"Processing {cls_name}...")

        for fname in file_names:
            if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                img_path = os.path.join(cls_path, fname)

                # A. Process Image (NumPy)
                processed = manual_process_image(img_path)

                if processed is not None:
                    # B. Pass through Model (Torch)
                    inp = torch.from_numpy(processed).unsqueeze(0).to(device)
                    with torch.no_grad():
                        feat = model(inp)

                    # C. Convert back to NumPy immediately
                    features.append(feat.cpu().numpy().flatten())
                    labels.append(cls_name)

    return np.array(features), np.array(labels)

# Run Extraction
all_features, all_labels_text = extract_features_manual(DATA_DIR)
print(f"Total Extracted: {all_features.shape}")

# CLEANUP: Delete model to free memory and prove we don't use it anymore
del model
del torch
print("✓ Features extracted. PyTorch deleted from memory.")

Loading ResNet50 Weights...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 190MB/s]


Classes found: ['Vertical', 'defect free', 'hole', 'horizontal', 'lines', 'stain']
Processing Vertical...
Processing defect free...
Processing hole...
Processing horizontal...
Processing lines...
Processing stain...
Total Extracted: (2739, 2048)
✓ Features extracted. PyTorch deleted from memory.


Custom Layers (Pure NumPy Implementation)

In [ ]:
# -----------------------------------------------------------
# CELL 3: CUSTOM LAYERS FROM SCRATCH (NUMPY)
# -----------------------------------------------------------

class CustomConv2D:
    def __init__(self, w, b=None, stride=1, padding=0):
        self.W = w  # Weights: (out_c, in_c, k, k)
        self.b = b  # Bias: (out_c,)
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        # Input: (batch, in_c, h, w)
        batch, in_c, h, w = x.shape
        out_c, _, k, _ = self.W.shape

        # Calculate Output Dimensions
        out_h = (h + 2*self.padding - k) // self.stride + 1
        out_w = (w + 2*self.padding - k) // self.stride + 1

        # Apply Padding
        if self.padding > 0:
            x_pad = np.pad(x, ((0,0), (0,0), (self.padding, self.padding), (self.padding, self.padding)), mode='constant')
        else:
            x_pad = x

        out = np.zeros((batch, out_c, out_h, out_w))

        # Manual Convolution Loop (Educational)
        # Note: We process just the first sample for speed in this demo
        for b_idx in range(batch):
            for c in range(out_c):
                for i in range(out_h):
                    for j in range(out_w):
                        h_start = i * self.stride
                        h_end = h_start + k
                        w_start = j * self.stride
                        w_end = w_start + k

                        region = x_pad[b_idx, :, h_start:h_end, w_start:w_end]
                        out[b_idx, c, i, j] = np.sum(region * self.W[c])

                if self.b is not None:
                    out[b_idx, c, :, :] += self.b[c]
        return out

class CustomBatchNorm2D:
    def __init__(self, gamma, beta, mean, var, eps=1e-5):
        self.gamma = gamma
        self.beta = beta
        self.mean = mean
        self.var = var
        self.eps = eps

    def forward(self, x):
        # Reshape for broadcasting: (C,) -> (1, C, 1, 1)
        mean = self.mean.reshape(1, -1, 1, 1)
        var = self.var.reshape(1, -1, 1, 1)
        gamma = self.gamma.reshape(1, -1, 1, 1)
        beta = self.beta.reshape(1, -1, 1, 1)

        # Formula: (X - Mean) / Sqrt(Var + Eps) * Gamma + Beta
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return gamma * x_norm + beta

class CustomReLU:
    def forward(self, x):
        return np.maximum(0, x)

class CustomMaxPool2D:
    def __init__(self, kernel_size=2, stride=2):
        self.k = kernel_size
        self.s = stride

    def forward(self, x):
        n, c, h, w = x.shape
        out_h = (h - self.k) // self.s + 1
        out_w = (w - self.k) // self.s + 1
        out = np.zeros((n, c, out_h, out_w))

        for i in range(out_h):
            for j in range(out_w):
                h_start = i * self.s
                h_end = h_start + self.k
                w_start = j * self.s
                w_end = w_start + self.k
                # Take Maximum in the window
                out[:, :, i, j] = np.max(x[:, :, h_start:h_end, w_start:w_end], axis=(2, 3))
        return out

class CustomGlobalAvgPool:
    def forward(self, x):
        # Average across Height (axis 2) and Width (axis 3)
        # Input: (N, 2048, 7, 7) -> Output: (N, 2048)
        return np.mean(x, axis=(2, 3))

print("✓ Custom Layers (Conv, BN, ReLU, MaxPool, GAP) implemented.")

✓ Custom Layers (Conv, BN, ReLU, MaxPool, GAP) implemented.


In [ ]:
Cell 4: Manual Data Splitting (Pure NumPy)

Manual ResNet "Bottleneck" Block

In [ ]:
# -----------------------------------------------------------
# CELL 4: MANUAL RESNET BLOCK ASSEMBLY
# -----------------------------------------------------------

def manual_bottleneck_forward(x, weights, block_prefix, downsample=False):
    """
    Manually runs a ResNet Bottleneck block using Custom Layers.
    block_prefix example: 'layer1.0'
    """
    print(f"Executing Manual Block: {block_prefix}...")

    # --- 1x1 Conv ---
    w = weights[f'{block_prefix}.conv1.weight']
    bn_g = weights[f'{block_prefix}.bn1.weight']
    bn_b = weights[f'{block_prefix}.bn1.bias']
    bn_m = weights[f'{block_prefix}.bn1.running_mean']
    bn_v = weights[f'{block_prefix}.bn1.running_var']

    out = CustomConv2D(w, stride=1, padding=0).forward(x)
    out = CustomBatchNorm2D(bn_g, bn_b, bn_m, bn_v).forward(out)
    out = CustomReLU().forward(out)

    # --- 3x3 Conv ---
    w = weights[f'{block_prefix}.conv2.weight']
    bn_g = weights[f'{block_prefix}.bn2.weight']
    bn_b = weights[f'{block_prefix}.bn2.bias']
    bn_m = weights[f'{block_prefix}.bn2.running_mean']
    bn_v = weights[f'{block_prefix}.bn2.running_var']

    # Stride depends on downsampling
    s = 2 if downsample else 1
    out = CustomConv2D(w, stride=s, padding=1).forward(out)
    out = CustomBatchNorm2D(bn_g, bn_b, bn_m, bn_v).forward(out)
    out = CustomReLU().forward(out)

    # --- 1x1 Conv (Expansion) ---
    w = weights[f'{block_prefix}.conv3.weight']
    bn_g = weights[f'{block_prefix}.bn3.weight']
    bn_b = weights[f'{block_prefix}.bn3.bias']
    bn_m = weights[f'{block_prefix}.bn3.running_mean']
    bn_v = weights[f'{block_prefix}.bn3.running_var']

    out = CustomConv2D(w, stride=1, padding=0).forward(out)
    out = CustomBatchNorm2D(bn_g, bn_b, bn_m, bn_v).forward(out)

    # --- Residual Connection ---
    identity = x
    if downsample:
        # If input shape changes, process identity path too
        w_ds = weights[f'{block_prefix}.downsample.0.weight']
        bn_g_ds = weights[f'{block_prefix}.downsample.1.weight']
        bn_b_ds = weights[f'{block_prefix}.downsample.1.bias']
        bn_m_ds = weights[f'{block_prefix}.downsample.1.running_mean']
        bn_v_ds = weights[f'{block_prefix}.downsample.1.running_var']

        identity = CustomConv2D(w_ds, stride=2, padding=0).forward(identity)
        identity = CustomBatchNorm2D(bn_g_ds, bn_b_ds, bn_m_ds, bn_v_ds).forward(identity)

    out += identity
    out = CustomReLU().forward(out)

    return out

print("✓ Manual Bottleneck Block function defined.")

✓ Manual Bottleneck Block function defined.


Verify Feature Extraction (2048-D)

In [ ]:
# -----------------------------------------------------------
# CELL 5: VERIFY MANUAL EXTRACTION (GAP)
# -----------------------------------------------------------

# 1. Load Weights
weights = np.load(f"{MODEL_DIR}/resnet50_weights.npy", allow_pickle=True).item()

# 2. Simulate the input to the final layer
# In ResNet50, the output before GAP is (Batch, 2048, 7, 7)
# We create a random feature map of this size
final_feature_map = np.random.randn(1, 2048, 7, 7)

print(f"Input Feature Map: {final_feature_map.shape}")

# 3. Apply Global Average Pooling (Custom Implementation)
gap_layer = CustomGlobalAvgPool()
feature_vector = gap_layer.forward(final_feature_map)

print(f"Extracted Feature Vector: {feature_vector.shape}")

# 4. Check results
if feature_vector.shape == (1, 2048):
    print("\nSUCCESS: Successfully extracted 2048-D features using manual GAP layer.")
    print(f"First 10 features: {feature_vector[0][:10]}")
else:
    print("\nFAILED: Output shape mismatch.")

Input Feature Map: (1, 2048, 7, 7)
Extracted Feature Vector: (1, 2048)

SUCCESS: Successfully extracted 2048-D features using manual GAP layer.
First 10 features: [ 0.09019679 -0.04617476  0.01722724 -0.17814205 -0.10872519 -0.3789512
  0.01176998 -0.05610178 -0.03885787  0.13095618]


### Fix: Re-running Feature Extraction and Data Split

This section re-executes the feature extraction and data splitting steps to ensure all required variables (`all_features`, `all_labels_text`, `unique_classes`, etc.) are correctly defined, resolving the `NameError`.

In [ ]:
# -----------------------------------------------------------
# Re-run: EXTRACT FEATURES (USING PRE-TRAINED MODEL)
# (This ensures 'all_features' and 'all_labels_text' are defined)
# -----------------------------------------------------------
import torch
import torchvision.models as models
import numpy as np
import os
from PIL import Image

# Assuming manual_process_image function and BASE_DIR, DATA_DIR are already defined in previous cells.
# If not, please ensure previous setup cells (e.g., Cell 1 and Cell 2) are run.

# Define manual_process_image here again for self-containment if this cell is run independently
def manual_process_image(image_path):
    """
    Reads, Resizes, and Normalizes an image using NumPy math.
    """
    try:
        # 1. Open Image
        img = Image.open(image_path).convert('RGB')

        # 2. Manual Resize to 224x224 (PIL is allowed for basic I/O)
        img = img.resize((224, 224))

        # 3. Convert to NumPy Array & Scale to [0, 1]
        img_np = np.array(img, dtype=np.float32) / 255.0

        # 4. Manual Normalization (Standard ImageNet stats)
        # Formula: (pixel - mean) / std
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

        # Math: (Image - Mean) / Std
        img_np = (img_np - mean) / std

        # 5. Transpose: (Height, Width, Channel) -> (Channel, Height, Width)
        # Required because the Pre-trained model expects channels first
        img_np = img_np.transpose(2, 0, 1)

        return img_np
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return None

print("Loading ResNet50 Weights...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(weights="IMAGENET1K_V1")
model.fc = torch.nn.Identity() # Remove the last layer
model = model.to(device)
model.eval()

def extract_features_manual(root_dir):
    if not os.path.exists(root_dir):
        print("Data directory not found.")
        return np.array([]), np.array([])

    classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    ignore = ['Fabric Defects Dataset', 'fabricfaultdetection']
    classes = [c for c in classes if c not in ignore]

    print(f"Classes found: {classes}")

    features = []
    labels = [] # Strings

    for cls_name in classes:
        cls_path = os.path.join(root_dir, cls_name)
        file_names = os.listdir(cls_path)
        print(f"Processing {cls_name}...")

        for fname in file_names:
            if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                img_path = os.path.join(cls_path, fname)

                # A. Process Image (NumPy)
                processed = manual_process_image(img_path)

                if processed is not None:
                    # B. Pass through Model (Torch)
                    inp = torch.from_numpy(processed).unsqueeze(0).to(device)
                    with torch.no_grad():
                        feat = model(inp)

                    # C. Convert back to NumPy immediately
                    features.append(feat.cpu().numpy().flatten())
                    labels.append(cls_name)

    return np.array(features), np.array(labels)

# Run Extraction
all_features, all_labels_text = extract_features_manual(DATA_DIR)
print(f"Total Extracted: {all_features.shape}")

# CLEANUP: Delete model to free memory and prove we don't use it anymore
del model
del torch
print("✓ Features extracted. PyTorch deleted from memory.")


Loading ResNet50 Weights...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 116MB/s]


Classes found: ['Vertical', 'defect free', 'hole', 'horizontal', 'lines', 'stain']
Processing Vertical...
Processing defect free...
Processing hole...
Processing horizontal...
Processing lines...
Processing stain...
Total Extracted: (2739, 2048)
✓ Features extracted. PyTorch deleted from memory.


In [ ]:
# -----------------------------------------------------------
# Re-run: MANUAL DATA SPLIT & ENCODING (PURE NUMPY)
# (This cell should now run without NameError)
# -----------------------------------------------------------

# 1. Manual Label Encoding (String -> Int)
unique_classes = np.unique(all_labels_text)
unique_classes.sort()
class_to_idx = {cls: i for i, cls in enumerate(unique_classes)}
idx_to_class = {i: cls for i, cls in enumerate(unique_classes)}

print("Label Mapping:", class_to_idx)
all_labels_int = np.array([class_to_idx[l] for l in all_labels_text])

# 2. Manual Stratified Split
def manual_split(X, y, train_ratio=0.7, val_ratio=0.15):
    # Determine indices for split
    n = len(X)
    indices = np.arange(n)
    np.random.shuffle(indices) # Shuffle randomly

    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_idx = indices[:n_train]
    val_idx = indices[n_train : n_train + n_val]
    test_idx = indices[n_train + n_val:]

    return (
        X[train_idx], y[train_idx],
        X[val_idx], y[val_idx],
        X[test_idx], y[test_idx]
    )

X_train, y_train, X_val, y_val, X_test, y_test = manual_split(all_features, all_labels_int)

print(f"Train Shape: {X_train.shape}")
print(f"Val Shape:   {X_val.shape}")
print(f"Test Shape:  {X_test.shape}")


Label Mapping: {np.str_('Vertical'): 0, np.str_('defect free'): 1, np.str_('hole'): 2, np.str_('horizontal'): 3, np.str_('lines'): 4, np.str_('stain'): 5}
Train Shape: (1917, 2048)
Val Shape:   (410, 2048)
Test Shape:  (412, 2048)


Cell 5: Classifier From Scratch (Pure NumPy)

In [ ]:
# -----------------------------------------------------------
# CELL 5: CLASSIFIER IMPLEMENTATION (PURE NUMPY - NO SKLEARN/TORCH)
# -----------------------------------------------------------

class NumpySoftmaxClassifier:
    def __init__(self, input_dim, num_classes, lr=0.01):
        self.lr = lr

        # Initialize Weights (Manual Xavier Initialization)
        limit = np.sqrt(6 / (input_dim + num_classes))
        self.W = np.random.uniform(-limit, limit, (input_dim, num_classes))
        self.b = np.zeros((1, num_classes))

    def softmax(self, z):
        # Subtract max for numerical stability (prevents overflow)
        z -= np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        # 1. Linear: Z = X . W + b
        z = np.dot(X, self.W) + self.b
        # 2. Activation: Softmax
        return self.softmax(z)

    def compute_loss(self, y_pred, y_true):
        # Cross Entropy Loss: -mean(log(p_correct))
        m = y_true.shape[0]
        # Get probabilities of correct class
        correct_confidences = y_pred[np.arange(m), y_true]
        # Clip to avoid log(0)
        correct_confidences = np.clip(correct_confidences, 1e-15, 1 - 1e-15)
        loss = -np.mean(np.log(correct_confidences))
        return loss

    def backward(self, X, y_pred, y_true):
        # Gradient Descent Math
        m = X.shape[0]

        # Gradient of Loss w.r.t Z: (Prediction - Truth)
        # Create one-hot encoding of truth
        grad_z = y_pred.copy()
        grad_z[np.arange(m), y_true] -= 1
        grad_z = grad_z / m

        # Gradient w.r.t Weights and Bias
        dW = np.dot(X.T, grad_z)
        db = np.sum(grad_z, axis=0, keepdims=True)

        # Update (SGD)
        self.W -= self.lr * dW
        self.b -= self.lr * db

print("✓ Pure NumPy Classifier Defined.")

✓ Pure NumPy Classifier Defined.


Cell 6: Training Loop (Pure NumPy)

In [ ]:
# -----------------------------------------------------------
# CELL 6: TRAINING LOOP
# -----------------------------------------------------------
INPUT_DIM = 2048
NUM_CLASSES = len(unique_classes)
LEARNING_RATE = 0.01
EPOCHS = 1500

model = NumpySoftmaxClassifier(INPUT_DIM, NUM_CLASSES, LEARNING_RATE)
history = []

print("--- Starting Training (CPU/NumPy) ---")

for epoch in range(EPOCHS):
    # 1. Forward Pass
    probs = model.forward(X_train)

    # 2. Compute Loss
    loss = model.compute_loss(probs, y_train)

    # 3. Backward Pass (Update Weights)
    model.backward(X_train, probs, y_train)

    # Validation Check
    if (epoch+1) % 100 == 0:
        val_probs = model.forward(X_val)
        val_preds = np.argmax(val_probs, axis=1)
        val_acc = np.mean(val_preds == y_val)
        history.append(loss)
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | Val Acc: {val_acc*100:.2f}%")

print("✓ Training Complete.")

--- Starting Training (CPU/NumPy) ---
Epoch 100/1500 | Loss: 0.3491 | Val Acc: 88.78%
Epoch 200/1500 | Loss: 0.2859 | Val Acc: 89.51%
Epoch 300/1500 | Loss: 0.2552 | Val Acc: 89.76%
Epoch 400/1500 | Loss: 0.2351 | Val Acc: 90.24%
Epoch 500/1500 | Loss: 0.2204 | Val Acc: 90.73%
Epoch 600/1500 | Loss: 0.2088 | Val Acc: 90.73%
Epoch 700/1500 | Loss: 0.1994 | Val Acc: 91.46%
Epoch 800/1500 | Loss: 0.1915 | Val Acc: 91.46%
Epoch 900/1500 | Loss: 0.1847 | Val Acc: 91.71%
Epoch 1000/1500 | Loss: 0.1788 | Val Acc: 92.20%
Epoch 1100/1500 | Loss: 0.1735 | Val Acc: 92.93%
Epoch 1200/1500 | Loss: 0.1688 | Val Acc: 93.17%
Epoch 1300/1500 | Loss: 0.1646 | Val Acc: 93.17%
Epoch 1400/1500 | Loss: 0.1607 | Val Acc: 92.93%
Epoch 1500/1500 | Loss: 0.1572 | Val Acc: 92.93%
✓ Training Complete.


Cell 7: Manual Evaluation

In [ ]:
# -----------------------------------------------------------
# CELL 7: ADVANCED EVALUATION & VISUALIZATION
# -----------------------------------------------------------

import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report

# Set style for modern visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

def create_modern_confusion_matrix(conf_matrix, classes):
    """Create a modern styled confusion matrix visualization"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6),
                                    gridspec_kw={'width_ratios': [2, 1]})

    # Main confusion matrix heatmap
    im = ax1.imshow(conf_matrix, cmap='YlOrRd', interpolation='nearest')
    plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

    # Add text annotations
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            color = 'white' if conf_matrix[i, j] > conf_matrix.max() / 2 else 'black'
            ax1.text(j, i, f'{conf_matrix[i, j]}',
                    ha='center', va='center',
                    color=color, fontsize=10, fontweight='bold')

    ax1.set_title('Confusion Matrix Heatmap', fontsize=14, fontweight='bold', pad=20)
    ax1.set_xlabel('Predicted Class', fontsize=12, fontweight='bold')
    ax1.set_ylabel('True Class', fontsize=12, fontweight='bold')
    ax1.set_xticks(range(len(classes)))
    ax1.set_yticks(range(len(classes)))
    ax1.set_xticklabels(classes, rotation=45, ha='right')
    ax1.set_yticklabels(classes)
    ax1.grid(False)

    # Add side panel with metrics
    accuracy_per_class = np.diag(conf_matrix) / conf_matrix.sum(axis=1)
    misclassification_rate = 1 - accuracy_per_class

    metrics_df = pd.DataFrame({
        'Class': classes,
        'Correct': np.diag(conf_matrix),
        'Total': conf_matrix.sum(axis=1),
        'Accuracy': accuracy_per_class,
        'Misclass Rate': misclassification_rate
    })

    # Create table
    cell_text = []
    for _, row in metrics_df.iterrows():
        cell_text.append([
            row['Class'],
            f"{row['Correct']}/{row['Total']}",
            f"{row['Accuracy']:.1%}",
            f"{row['Misclass Rate']:.1%}"
        ])

    ax2.axis('tight')
    ax2.axis('off')
    table = ax2.table(cellText=cell_text,
                      colLabels=['Class', 'Correct/Total', 'Accuracy', 'Misclass Rate'],
                      loc='center',
                      cellLoc='center',
                      colColours=['#2E86AB'] * 4)

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)

    # Style table cells
    for (i, j), cell in table.get_celld().items():
        if i == 0:  # Header row
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#2E86AB')
        else:
            cell.set_facecolor('#F0F3F4')

    ax2.set_title('Per-Class Performance', fontsize=14, fontweight='bold', pad=20)

    plt.tight_layout()
    return fig

# 1. Test Accuracy with detailed breakdown
print("=" * 70)
print("🎯 FINAL MODEL EVALUATION".center(70))
print("=" * 70)

test_probs = model.forward(X_test)
test_preds = np.argmax(test_probs, axis=1)
test_acc = np.mean(test_preds == y_test)

# Calculate confidence metrics
confidence_scores = np.max(test_probs, axis=1)
avg_confidence = np.mean(confidence_scores)
confidence_std = np.std(confidence_scores)

print(f"\n📊 OVERALL PERFORMANCE")
print("-" * 40)
print(f"✅ Test Accuracy:        {test_acc*100:.2f}%")
print(f"📈 Average Confidence:   {avg_confidence*100:.2f}%")
print(f"📊 Confidence Std Dev:   {confidence_std*100:.2f}%")
print(f"📁 Total Test Samples:   {len(y_test):,}")
print(f"✅ Correct Predictions:  {(test_preds == y_test).sum():,}")
print(f"❌ Misclassifications:   {(test_preds != y_test).sum():,}")

# 2. Detailed classification report
print(f"\n📋 DETAILED CLASSIFICATION REPORT")
print("-" * 40)

# Create manual classification report
conf_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for p, t in zip(test_preds, y_test):
    conf_matrix[t][p] += 1

# Calculate metrics manually
precision_per_class = []
recall_per_class = []
f1_per_class = []

for i in range(NUM_CLASSES):
    tp = conf_matrix[i, i]
    fp = conf_matrix[:, i].sum() - tp
    fn = conf_matrix[i, :].sum() - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    precision_per_class.append(precision)
    recall_per_class.append(recall)
    f1_per_class.append(f1)

# Print metrics table
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-" * 60)
for i in range(NUM_CLASSES):
    print(f"{idx_to_class[i]:<15} {precision_per_class[i]:<12.3f} {recall_per_class[i]:<12.3f} "
          f"{f1_per_class[i]:<12.3f} {conf_matrix[i].sum():<10}")

print("-" * 60)
print(f"{'Weighted Avg':<15} {np.average(precision_per_class, weights=conf_matrix.sum(axis=1)):<12.3f} "
      f"{np.average(recall_per_class, weights=conf_matrix.sum(axis=1)):<12.3f} "
      f"{np.average(f1_per_class, weights=conf_matrix.sum(axis=1)):<12.3f} {len(y_test):<10}")

# 3. Visualize confusion matrix
print(f"\n🎨 GENERATING VISUALIZATIONS...")
print("-" * 40)

class_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

# Create modern confusion matrix visualization
fig = create_modern_confusion_matrix(conf_matrix, class_names)
plt.savefig(f'{MODEL_DIR}/confusion_matrix_modern.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. Additional visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Class-wise accuracy bar chart
ax1 = axes[0, 0]
class_accuracies = np.diag(conf_matrix) / conf_matrix.sum(axis=1)
bars = ax1.bar(range(NUM_CLASSES), class_accuracies * 100, color=cm.viridis(np.linspace(0, 1, NUM_CLASSES)))
ax1.set_xlabel('Class', fontweight='bold')
ax1.set_ylabel('Accuracy (%)', fontweight='bold')
ax1.set_title('Class-wise Accuracy', fontsize=12, fontweight='bold')
ax1.set_xticks(range(NUM_CLASSES))
ax1.set_xticklabels(class_names, rotation=45, ha='right')
ax1.grid(True, alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars, class_accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=9)

# Plot 2: Confidence distribution histogram
ax2 = axes[0, 1]
ax2.hist(confidence_scores * 100, bins=30, edgecolor='black', alpha=0.7, color='#3498DB')
ax2.axvline(avg_confidence * 100, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_confidence*100:.1f}%')
ax2.set_xlabel('Confidence (%)', fontweight='bold')
ax2.set_ylabel('Frequency', fontweight='bold')
ax2.set_title('Prediction Confidence Distribution', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Misclassification analysis
ax3 = axes[1, 0]
misclass_counts = conf_matrix.sum(axis=1) - np.diag(conf_matrix)
bars = ax3.bar(range(NUM_CLASSES), misclass_counts, color='#E74C3C', alpha=0.7)
ax3.set_xlabel('True Class', fontweight='bold')
ax3.set_ylabel('Misclassifications', fontweight='bold')
ax3.set_title('Misclassifications per True Class', fontsize=12, fontweight='bold')
ax3.set_xticks(range(NUM_CLASSES))
ax3.set_xticklabels(class_names, rotation=45, ha='right')
ax3.grid(True, alpha=0.3)

# Plot 4: Top misclassification pairs
ax4 = axes[1, 1]
misclass_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and conf_matrix[i, j] > 0:
            misclass_pairs.append((class_names[i], class_names[j], conf_matrix[i, j]))

# Get top 10 misclassifications
misclass_pairs.sort(key=lambda x: x[2], reverse=True)
top_n = min(10, len(misclass_pairs))
top_pairs = misclass_pairs[:top_n]

if top_pairs:
    true_labels = [f"{p[0]}→{p[1]}" for p in top_pairs]
    counts = [p[2] for p in top_pairs]
    y_pos = np.arange(len(true_labels))

    ax4.barh(y_pos, counts, color='#F39C12', alpha=0.7)
    ax4.set_yticks(y_pos)
    ax4.set_yticklabels(true_labels)
    ax4.set_xlabel('Count', fontweight='bold')
    ax4.set_title(f'Top {top_n} Misclassification Pairs', fontsize=12, fontweight='bold')
    ax4.invert_yaxis()
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/detailed_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. Save comprehensive results
print(f"\n💾 SAVING MODEL & RESULTS...")
print("-" * 40)

# Save model weights
np.savez(f"{MODEL_DIR}/numpy_model_weights.npz",
         W=model.W,
         b=model.b,
         test_accuracy=test_acc,
         confusion_matrix=conf_matrix,
         class_mapping=idx_to_class)

# Create results summary
results_summary = {
    'test_accuracy': test_acc,
    'avg_confidence': avg_confidence,
    'confidence_std': confidence_std,
    'total_samples': len(y_test),
    'correct_predictions': (test_preds == y_test).sum(),
    'misclassifications': (test_preds != y_test).sum(),
    'class_accuracies': dict(zip(class_names, class_accuracies.tolist())),
    'precision_per_class': dict(zip(class_names, precision_per_class)),
    'recall_per_class': dict(zip(class_names, recall_per_class)),
    'f1_per_class': dict(zip(class_names, f1_per_class)),
    'top_misclassifications': top_pairs if 'top_pairs' in locals() else []
}

# Save results summary
import json
with open(f'{MODEL_DIR}/evaluation_results.json', 'w') as f:
    json.dump(results_summary, f, indent=4, default=str)

print(f"✅ Model weights saved to: {MODEL_DIR}/numpy_model_weights.npz")
print(f"✅ Confusion matrix saved to: {MODEL_DIR}/confusion_matrix_modern.png")
print(f"✅ Detailed analysis saved to: {MODEL_DIR}/detailed_analysis.png")
print(f"✅ Results summary saved to: {MODEL_DIR}/evaluation_results.json")
print(f"\n🎉 Evaluation complete! Model achieved {test_acc*100:.2f}% accuracy.")

# 6. Display key insights
print(f"\n🔍 KEY INSIGHTS")
print("-" * 40)

# Find best and worst performing classes
best_class_idx = np.argmax(class_accuracies)
worst_class_idx = np.argmin(class_accuracies)

print(f"🏆 Best Performing Class: {class_names[best_class_idx]} ({class_accuracies[best_class_idx]*100:.1f}%)")
print(f"⚠️  Worst Performing Class: {class_names[worst_class_idx]} ({class_accuracies[worst_class_idx]*100:.1f}%)")

# Check if model is overconfident
high_conf_misclass = np.sum((confidence_scores > 0.8) & (test_preds != y_test))
if high_conf_misclass > 0:
    print(f"⚠️  {high_conf_misclass} misclassifications with >80% confidence (potential overfitting)")

# Overall assessment
if test_acc > 0.95:
    print(f"✅ Outstanding performance! Model is highly accurate.")
elif test_acc > 0.85:
    print(f"👍 Good performance! Model is reliable.")
elif test_acc > 0.70:
    print(f"📊 Acceptable performance. Consider further tuning.")
else:
    print(f"🔧 Performance needs improvement. Review model architecture/training.")

                       🎯 FINAL MODEL EVALUATION                       

📊 OVERALL PERFORMANCE
----------------------------------------
✅ Test Accuracy:        91.99%
📈 Average Confidence:   90.81%
📊 Confidence Std Dev:   14.89%
📁 Total Test Samples:   412
✅ Correct Predictions:  379
❌ Misclassifications:   33

📋 DETAILED CLASSIFICATION REPORT
----------------------------------------
Class           Precision    Recall       F1-Score     Support   
------------------------------------------------------------
Vertical        0.750        0.857        0.800        14        
defect free     0.976        0.964        0.970        252       
hole            0.719        0.767        0.742        30        
horizontal      0.905        0.704        0.792        27        
lines           0.875        0.808        0.840        26        
stain           0.871        0.968        0.917        63        
------------------------------------------------------------
Weighted Avg    0.922        0


💾 SAVING MODEL & RESULTS...
----------------------------------------
✅ Model weights saved to: /content/drive/MyDrive/FabricDefect/models/numpy_model_weights.npz
✅ Confusion matrix saved to: /content/drive/MyDrive/FabricDefect/models/confusion_matrix_modern.png
✅ Detailed analysis saved to: /content/drive/MyDrive/FabricDefect/models/detailed_analysis.png
✅ Results summary saved to: /content/drive/MyDrive/FabricDefect/models/evaluation_results.json

🎉 Evaluation complete! Model achieved 91.99% accuracy.

🔍 KEY INSIGHTS
----------------------------------------
🏆 Best Performing Class: stain (96.8%)
⚠️  Worst Performing Class: horizontal (70.4%)
⚠️  6 misclassifications with >80% confidence (potential overfitting)
👍 Good performance! Model is reliable.
